# Pixar's USD in Python

In addition to the official [USD tutorials](https://openusd.org/release/tut_usd_tutorials.html), this [Houdini tutorial](https://github.com/kiryha/Houdini/wiki/Pixar-USD-Python-API#create-geometry-with-python) is an interesting reference and was used for this notebook.


## Pre-defined Sphere

In [ ]:
from pxr import Usd, UsdGeom

stage = Usd.Stage.CreateNew("pixar-sphere.usda")
UsdGeom.Xform.Define(stage, "/Root")
UsdGeom.Sphere.Define(stage, "/Root/pixar_sphere")
stage.GetRootLayer().Save()

## Mesh

In [ ]:
from pxr import Usd, UsdGeom

# Create USD
stage = Usd.Stage.CreateNew("mesh.usda")

# Build mesh object
_ = UsdGeom.Xform.Define(stage, "/Root")
mesh = UsdGeom.Mesh.Define(stage, "/Root/Mesh")

# Build mesh geometry. Here polygon creation magic should happen
geometry_data = {
    "points": [(-1, 0, 1), (1, 0, 1), (1, 0, -1), (-1, 0, -1)],
    "face_vertex_counts": [4],
    "face_vertex_indices": [0, 1, 2, 3],
}

# Set mesh attributes
mesh.GetPointsAttr().Set(geometry_data["points"])
mesh.GetFaceVertexCountsAttr().Set(geometry_data["face_vertex_counts"])
mesh.GetFaceVertexIndicesAttr().Set(geometry_data["face_vertex_indices"])

# Save USD
stage.GetRootLayer().Save()

## Procedural

In [ ]:
import math

from pxr import Usd, UsdGeom


def cone(resolution: int) -> dict[str, list]:
    """Create polygonal cone."""
    points = []  # List of point positions
    face_vertex_counts = []  # List of vertex count per face
    face_vertex_indices = []  # List of vertex indices

    # Create cone points
    for point in range(resolution):
        angle = 2.0 * 3.14 * point / resolution

        x = math.cos(angle)
        z = math.sin(angle)
        points.append((x, 0, z))

    # Add tip
    points.append((0, 2, 0))

    # Crete cone faces
    for point in range(resolution):
        triangle = [point, (point + 1) % resolution, resolution]
        face_vertex_indices.extend(triangle)
        face_vertex_counts.append(3)

    return {
        "points": points,
        "face_vertex_counts": face_vertex_counts,
        "face_vertex_indices": face_vertex_indices,
    }


def create_geometry() -> None:
    """Procedurally create geometry and save it to the USDA file."""
    # Create USD
    stage = Usd.Stage.CreateNew("cone.usda")

    # Build mesh object
    _ = UsdGeom.Xform.Define(stage, "/Root")
    mesh = UsdGeom.Mesh.Define(stage, "/Root/Cone")

    # Build cone geometry.
    geometry_data = cone(12)

    # Set mesh attributes
    mesh.GetPointsAttr().Set(geometry_data["points"])
    mesh.GetFaceVertexCountsAttr().Set(geometry_data["face_vertex_counts"])
    mesh.GetFaceVertexIndicesAttr().Set(geometry_data["face_vertex_indices"])

    # Set orientation and subdivisionScheme
    mesh.CreateOrientationAttr().Set(UsdGeom.Tokens.leftHanded)
    mesh.CreateSubdivisionSchemeAttr().Set("none")

    # Save USD
    stage.GetRootLayer().Save()


create_geometry()

## Procedural Sphere

In [ ]:
from pxr import Usd, UsdGeom


def get_cartesian_position(
    h_angle: float,
    v_angle: float,
) -> tuple[float, float, float]:
    """Convert angles to cartesian coordinates."""
    return (
        math.sin(v_angle) * math.cos(h_angle),
        math.sin(v_angle) * math.sin(h_angle),
        math.cos(v_angle),
    )


def sphere(h_points: int, v_points: int) -> dict[str, list]:
    """Create polygonal sphere."""
    points = []  # List of point positions
    face_vertex_counts = []  # List of vertex count per face
    face_vertex_indices = []  # List of vertex indices

    # Crate sphere points
    points.append((0, 0, 1))  # Top pole

    for v_point in range(1, v_points - 1):  # Range excludes poles
        v_angle = v_point * 3.14 / (v_points - 1)

        for h_point in range(h_points):
            h_angle = 2 * h_point * 3.14 / h_points

            position = get_cartesian_position(h_angle, v_angle)
            points.append(position)

    points.append((0, 0, -1))  # Bottom pole

    # Create sphere faces
    # Top pole faces
    top_pole_index = 0
    first_row_start = 1
    for h_point in range(h_points):
        next_point = (h_point + 1) % h_points
        face_vertex_indices.extend(
            [top_pole_index, first_row_start + next_point, first_row_start + h_point],
        )
        face_vertex_counts.append(3)

    # Main body faces (quads)
    for v_point in range(1, v_points - 2):
        row_start = 1 + (v_point - 1) * h_points
        next_row_start = row_start + h_points
        for h_point in range(h_points):
            next_point = (h_point + 1) % h_points
            face_vertex_indices.extend(
                [
                    row_start + h_point,
                    row_start + next_point,
                    next_row_start + next_point,
                    next_row_start + h_point,
                ],
            )
            face_vertex_counts.append(4)

    # Bottom pole faces
    bottom_pole_index = len(points) - 1
    last_row_start = 1 + (v_points - 3) * h_points
    for h_point in range(h_points):
        next_point = (h_point + 1) % h_points
        face_vertex_indices.extend(
            [bottom_pole_index, last_row_start + h_point, last_row_start + next_point],
        )
        face_vertex_counts.append(3)

    return {
        "points": points,
        "face_vertex_counts": face_vertex_counts,
        "face_vertex_indices": face_vertex_indices,
    }


def create_geometry() -> None:
    """Procedurally create geometry and save it to the USDA file."""
    # Create USD
    stage = Usd.Stage.CreateNew("sphere.usda")

    # Build mesh object
    _ = UsdGeom.Xform.Define(stage, "/Root")
    mesh = UsdGeom.Mesh.Define(stage, "/Root/Sphere")

    # Build cone geometry.
    geometry_data = sphere(8, 6)

    # Set mesh attributes
    mesh.GetPointsAttr().Set(geometry_data["points"])
    mesh.GetFaceVertexCountsAttr().Set(geometry_data["face_vertex_counts"])
    mesh.GetFaceVertexIndicesAttr().Set(geometry_data["face_vertex_indices"])

    # Set orientation and subdivisionScheme
    mesh.CreateOrientationAttr().Set(UsdGeom.Tokens.leftHanded)
    mesh.CreateSubdivisionSchemeAttr().Set("none")

    # Save USD
    stage.GetRootLayer().Save()


create_geometry()

## Procedural Plane

In [ ]:
from pxr import Usd, UsdGeom


def plane(row_points: int, col_points: int) -> dict[str, list]:
    """Create procedural plane with custom number of rows and columns and size of 2."""
    points = []  # List of point positions
    face_vertex_counts = []  # List of vertex count per face
    face_vertex_indices = []  # List of vertex indices

    # Spacing between points
    width = 2
    height = 2
    row_spacing = height / (row_points - 1)
    col_spacing = width / (col_points - 1)

    # Generate points for the grid
    for row_point in range(row_points):
        for column_point in range(col_points):
            x = column_point * col_spacing - width / 2
            z = row_point * row_spacing - height / 2
            points.append((x, 0, z))

    # Define faces using the indices of the grid points
    for row_point in range(row_points - 1):
        for column_point in range(col_points - 1):
            # Calculate the indices of the corners of the cell
            top_left = row_point * col_points + column_point
            top_right = top_left + 1
            bottom_left = top_left + col_points
            bottom_right = bottom_left + 1

            # Define the face using the indices of the 4 corners
            face_vertex_indices.extend([top_left, top_right, bottom_right, bottom_left])
            face_vertex_counts.append(4)

    return {
        "points": points,
        "face_vertex_counts": face_vertex_counts,
        "face_vertex_indices": face_vertex_indices,
    }


def create_geometry() -> None:
    """Procedurally create geometry and save it to the USDA file."""
    # Create USD
    stage = Usd.Stage.CreateNew("plane.usda")

    # Build mesh object
    _ = UsdGeom.Xform.Define(stage, "/Root")
    mesh = UsdGeom.Mesh.Define(stage, "/Root/Plane")

    # Build cone geometry.
    geometry_data = plane(8, 8)

    # Set mesh attributes
    mesh.GetPointsAttr().Set(geometry_data["points"])
    mesh.GetFaceVertexCountsAttr().Set(geometry_data["face_vertex_counts"])
    mesh.GetFaceVertexIndicesAttr().Set(geometry_data["face_vertex_indices"])

    # Set orientation and subdivisionScheme
    mesh.CreateOrientationAttr().Set(UsdGeom.Tokens.leftHanded)
    mesh.CreateSubdivisionSchemeAttr().Set("none")

    # Save USD
    stage.GetRootLayer().Save()


create_geometry()

## Extruding Polygons

In [ ]:
import copy

from pxr import Usd, UsdGeom


def extrude_polygon() -> dict[str, list]:
    """Define source polygonal plane with hardcoded values.

    Perform extrusion operation on it returning new data.

    """
    # Define source polygon
    points = [(-1, 0, 1), (1, 0, 1), (1, 0, -1), (-1, 0, -1)]
    face_vertex_counts = [4]
    face_vertex_indices = [0, 1, 2, 3]

    source_polygon = {
        "points": points,
        "face_vertex_counts": face_vertex_counts,
        "face_vertex_indices": face_vertex_indices,
    }

    # Copy source polygon data to a new variable
    extruded_polygon = copy.deepcopy(source_polygon)

    # Define extrusion distance
    extrude_distance = 2

    # Loop source face points and create new points with shifted positions
    for point in source_polygon["points"]:
        extruded_point = (point[0], point[1] + extrude_distance, point[2])
        extruded_polygon["points"].append(extruded_point)

    # Add face for each pair of old/new points (edges)
    source_points = len(source_polygon["points"])
    for index in range(source_points):
        lower_left = index
        lower_right = (index + 1) % source_points
        upper_right = ((index + 1) % source_points) + source_points
        upper_left = index + source_points

        quad = [upper_left, upper_right, lower_right, lower_left]

        extruded_polygon["face_vertex_indices"].extend(quad)
        extruded_polygon["face_vertex_counts"].append(4)

    # Add top face
    extruded_polygon["face_vertex_indices"].extend([7, 6, 5, 4])
    extruded_polygon["face_vertex_counts"].append(4)

    return extruded_polygon


def create_geometry() -> None:
    """Procedurally create geometry and save it to the USDA file."""
    stage = Usd.Stage.CreateNew("extruded-plane.usda")

    # Build mesh object
    UsdGeom.Xform.Define(stage, "/Root")
    mesh = UsdGeom.Mesh.Define(stage, "/Root/ExtrudedPlane")

    geometry_data = extrude_polygon()

    mesh.GetPointsAttr().Set(geometry_data["points"])
    mesh.GetFaceVertexCountsAttr().Set(geometry_data["face_vertex_counts"])
    mesh.GetFaceVertexIndicesAttr().Set(geometry_data["face_vertex_indices"])

    # Set orientation and subdivisionScheme
    mesh.CreateOrientationAttr().Set(UsdGeom.Tokens.leftHanded)
    mesh.CreateSubdivisionSchemeAttr().Set("none")

    stage.GetRootLayer().Save()


create_geometry()

# Attempt at Geospatial Application

In [ ]:
import numpy as np
import pyproj
from pxr import Usd, UsdGeom

from osm_scene.constants import GEOD_WGS84

minx = 13.3568753
miny = 52.5181898
maxx = 13.3763515
maxy = 52.5275873

# top-left > top-right > bottom-right
(x_az, y_az), _, distance_m = GEOD_WGS84.inv(
    [minx, minx],
    [maxy, maxy],
    [maxx, minx],
    [maxy, miny],
    return_back_azimuth=False,
)

x_m, y_m = np.ceil(distance_m).astype(int)

# could add some buffers

x_int = GEOD_WGS84.fwd_intermediate(minx, maxy, x_az, npts=x_m, del_s=1.0)
y_int = GEOD_WGS84.fwd_intermediate(minx, maxy, y_az, npts=y_m, del_s=1.0)

mesh_lons, mesh_lats = np.meshgrid(x_int.lons, y_int.lats, copy=False)
mesh_ele = np.zeros(mesh_lons.shape)

faces_n_lats = mesh_ele.shape[0] - 1
faces_n_lons = mesh_ele.shape[1] - 1

i = np.tile(
    np.tile([0, 1, 1, 0], (faces_n_lats, 1))
    + np.arange(0, faces_n_lats)[:, np.newaxis],
    (1, faces_n_lons),
)

j = np.tile(
    (
        np.tile([0, 0, 1, 1], (faces_n_lons, 1))
        + np.arange(0, faces_n_lons)[:, np.newaxis]
    ).flatten(),
    (faces_n_lats, 1),
)

LLA_TO_ECEF = pyproj.Transformer.from_crs(
    {"proj": "latlon", "ellps": "WGS84", "datum": "WGS84"},
    {"proj": "geocent", "ellps": "WGS84", "datum": "WGS84"},
)

x, y, z = LLA_TO_ECEF.transform(
    mesh_lons.flatten(),
    mesh_lats.flatten(),
    mesh_ele.flatten(),
)

stage = Usd.Stage.CreateNew("sandbox.usda")
UsdGeom.Xform.Define(stage, "/Root")
mesh = UsdGeom.Mesh.Define(stage, "/Root/Ground")

# transform with quaternion / full transform so the result is flat on xy plane

mesh.GetPointsAttr().Set(
    list(map(list, zip(x - x.min(), y - y.min(), z - z.min(), strict=False))),
)
mesh.GetFaceVertexCountsAttr().Set(np.ones(faces_n_lats * faces_n_lons) * 4)
mesh.GetFaceVertexIndicesAttr().Set(
    np.ravel_multi_index((i.flatten(), j.flatten()), mesh_ele.shape),
)

stage.GetRootLayer().Save()